# 🚀 TACYOLO: 4-Pillar Universal Detection (Drones + CCTV + OpenImages + Thermal)

### Two-Stage Decoupled Workflow (Saves 100% of Expensive GPU Credits):
1. **Stage 1 [CPU Mode]**: Download all 4 pillars directly to Google Drive & slice into 640x640 tiles. (Zero GPU used!).
2. **Stage 2 [A100 GPU Mode]**: Switch to A100 GPU and train directly from Google Drive with **zero copy overhead**.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/TACYOLO')
for folder in ['working_testing', 'raw_data', 'tiled_batches', 'trained_weights', 'calibration_data', 'metrics_logs']:
    (DRIVE_ROOT / folder).mkdir(parents=True, exist_ok=True)
    print(f"Ready: {DRIVE_ROOT / folder}")


## 2. Setup Kaggle Credentials

In [ ]:
import os
from pathlib import Path

kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(parents=True, exist_ok=True)
(kaggle_dir / 'access_token').write_text('KGAT_bdd2506def33150b9b74a96cfa4a067d')
(kaggle_dir / 'kaggle.json').write_text('{"token": "KGAT_bdd2506def33150b9b74a96cfa4a067d"}')
os.chmod(str(kaggle_dir / 'access_token'), 0o600)
os.chmod(str(kaggle_dir / 'kaggle.json'), 0o600)
print("Kaggle credentials installed successfully.")


## 3. Clone Repository & Install Dependencies

In [ ]:
import os
if not os.path.exists('/content/tacyolo_repo'):
    !git clone https://github.com/devansh3108-2/tacyolo.git /content/tacyolo_repo
else:
    %cd /content/tacyolo_repo
    !git pull origin main

%cd /content/tacyolo_repo
!pip install -q ultralytics kagglehub opencv-python onnx onnxruntime


## 4A. [RUN ON CPU RUNTIME] Download & Tile All 4 Pillars into Drive (Zero GPU Waste)
- Downloads Drones (DOTA/VisDrone), CCTV (BDD100K), OpenImages V7, and Thermal FLIR directly to Drive.
- Slices into 640x640 tiles with bounding-box re-projection.
- Purges raw archives to keep Drive clean.
- **NO GPU NEEDED! Save your expensive A100 credits for training!**

In [ ]:
# 1. Ensure Drive is mounted
import os
from pathlib import Path
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

# 2. Ensure repository is present
if not os.path.exists('/content/tacyolo_repo'):
    !git clone https://github.com/devansh3108-2/tacyolo.git /content/tacyolo_repo
else:
    %cd /content/tacyolo_repo
    !git pull origin main

%cd /content/tacyolo_repo
!pip install -q ultralytics kagglehub opencv-python onnx onnxruntime

# 3. Run CPU-Only Pre-Tiling Stage (Skips finished datasets automatically)
!python scripts/prepare_tiles_cpu.py \
    --drive-root /content/drive/MyDrive/TACYOLO \
    --tile-size 640


## 4B. [SWITCH TO A100 GPU RUNTIME] Train Directly from Drive (Zero Copy Time)
1. In menu, switch runtime to **A100 GPU** + **High RAM ON**.
2. Run this cell: It reads the tiles directly from Google Drive with **ZERO copy overhead**!
3. Trains at maximum A100 speed and saves `best.pt` directly to Google Drive.

In [ ]:
# 1. Ensure Drive is mounted
import os
from pathlib import Path
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

# 2. Ensure repository is present
if not os.path.exists('/content/tacyolo_repo'):
    !git clone https://github.com/devansh3108-2/tacyolo.git /content/tacyolo_repo
else:
    %cd /content/tacyolo_repo
    !git pull origin main

%cd /content/tacyolo_repo
!pip install -q ultralytics kagglehub opencv-python onnx onnxruntime

# 3. Train directly from Google Drive on A100 GPU (Zero copy overhead)
!python scripts/train_gpu_direct.py \
    --drive-root /content/drive/MyDrive/TACYOLO \
    --epochs 100 \
    --batch 32 \
    --imgsz 640 \
    --model yolo11s.pt


## 5. [AFTER TRAINING] Harden INT8 Quantization & Export Edge Engine
Build a TensorRT calibration cache on operational footage for 250+ FPS real-time speed.

In [ ]:
!python -m tacyolo.runtime.quantize \
    --weights /content/drive/MyDrive/TACYOLO/trained_weights/best.pt \
    --calib-data /content/drive/MyDrive/TACYOLO/calibration_data \
    --output-dir /content/drive/MyDrive/TACYOLO/trained_weights \
    --imgsz 640


## 6. Inspect Final Model Artifacts on Google Drive

In [ ]:
!ls -lh /content/drive/MyDrive/TACYOLO/trained_weights
